# The Vibe Tax — LiveCodeBench pipeline (end to end)

Run the whole functional-code-evaluation pipeline in one place: **problems → tests → prompts → model responses → extraction & scoring → analysis.** The code-extraction step (`score_lcb.py :: extract_solution`) is the subject of the extraction-artifacts paper; this notebook lets you see it and run it against real data.

**How to use in Colab**
1. `Runtime → Run all`, or run cells top to bottom.
2. Optional secrets (`🔑` in the left sidebar → *Secrets*, or you'll be prompted): `HF_TOKEN` (to rebuild LCB tests), `OPENAI_API_KEY` / `ANTHROPIC_API_KEY` / `CODESTRAL_API_KEY` (only if you re-query models), `GITHUB_TOKEN` (only if the repo is private).
3. **By default nothing costs money** — the model responses are already committed, so scoring re-runs on them. The API stage is off behind a flag.

> The only file the repo does **not** ship is `lcb_tests.jsonl` (LCB tests are HF-gated). The notebook rebuilds it from HuggingFace (needs `HF_TOKEN` + dataset access), or you can upload your local copy.

## Data-flow graph (file → script → file)

```
                         [HuggingFace: livecodebench/code_generation_lite]
                                    |                      |
                    prepare_lcb.py  |                      |  extract_lcb_tests.py
                                    v                      v
                          lcb_problems.jsonl         lcb_tests.jsonl        (both LOCAL: HF-gated)
                                    |                      |
              generate_lcb_prompts.py                      |
                                    v                      |
                          lcb_v3_prompts.json              |
                                    |                      |
                       run_vibe_tax.py  (API calls)        |
                                    v                      |
                        lcb_v3_responses.json              |
                                    |                      |
                                    +----------+-----------+
                                               v
                                        score_lcb.py   <-- the extractor lives here
                                               v
                          lcb_scored.json  +  lcb_scored_stats.json
                                               |
                    +--------------------------+---------------------------+
                    v                          v                           v
             mcnemar_lcb.py         full_partial_credit.py        analyze_lcb_capability.py
                    v                          v                           v
        lcb_mcnemar_stats.json   full_partial_credit.json      lcb_capability_stats.json
                                               |
                                               v
                                   build_breakdown.py  (repo root)
                                               v
                              The_Vibe_Tax_Problem_Breakdown.xlsx
```

## Stage table

| # | Script | Reads | Writes | Needs | In this notebook |
|---|--------|-------|--------|-------|------------------|
| 1 | `prepare_lcb.py` | HuggingFace LCB | `lcb_problems.jsonl` | HF_TOKEN | **committed** — optional rebuild |
| 2 | `extract_lcb_tests.py` | HuggingFace LCB | `lcb_tests.jsonl` | HF_TOKEN | rebuilt (gitignored) or upload |
| 3 | `generate_lcb_prompts.py` | `lcb_problems.jsonl` | `lcb_v3_prompts.json` | — (mock backend) | rebuilt, keyless |
| 4 | `run_vibe_tax.py` | `lcb_v3_prompts.json` | `lcb_v3_responses.json` | API keys ($) | **committed** — off by default |
| 5 | `score_lcb.py` | tests + responses + problems | `lcb_scored.json` (+stats) | — | **runs the extractor** |
| 6a | `mcnemar_lcb.py` | `lcb_scored.json` | `lcb_mcnemar_stats.json` | — | runs |
| 6b | `full_partial_credit.py` | responses + tests | `full_partial_credit.json` | — (heavy) | optional |
| 6c | `analyze_lcb_capability.py` | `lcb_scored.json` | `lcb_capability_stats.json` | — | optional |
| 6d | `build_breakdown.py` | scored + responses + problems | `.xlsx` workbook | — | optional |

### 0 · Install dependencies

In [ ]:
!pip -q install openai anthropic huggingface_hub openpyxl

### 0.1 · Clone the repo and enter the pipeline directory
If the repo is private, add a `GITHUB_TOKEN` secret (a GitHub PAT with `repo` read).

In [ ]:
import os
BRANCH = "claude/blissful-hopper-dy0rvr"
REPO   = "jamelski1/the-vibe-tax"

tok = None
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")   # optional; only for a private repo
except Exception:
    tok = os.environ.get("GITHUB_TOKEN")

url = f"https://{tok}@github.com/{REPO}.git" if tok else f"https://github.com/{REPO}.git"
if not os.path.isdir("the-vibe-tax"):
    !git clone --quiet --branch {BRANCH} {url}
else:
    print("repo already cloned")
%cd the-vibe-tax/data/vibe_tax_lcb
import sys; sys.path.insert(0, ".")
!git log --oneline -1

### 0.2 · Configure secrets (optional)
Pulls from Colab *Secrets* if present, otherwise prompts. Skip any you don't need — scoring the committed responses needs **none** of these.

In [ ]:
import os, getpass
def _secret(name, prompt_if_missing=False):
    v = None
    try:
        from google.colab import userdata
        v = userdata.get(name)
    except Exception:
        v = os.environ.get(name)
    if not v and prompt_if_missing:
        v = getpass.getpass(f"{name} (leave blank to skip): ") or None
    if v:
        os.environ[name] = v
    return v

have = {k: bool(_secret(k)) for k in
        ("HF_TOKEN","OPENAI_API_KEY","ANTHROPIC_API_KEY","CODESTRAL_API_KEY")}
print("secrets present:", have)
print("\nNothing above is required to score the committed responses (stage 5).")

### Stage 1 · `prepare_lcb.py` → `lcb_problems.jsonl`  *(optional — already committed)*
The 167-problem file ships with the repo. Only rebuild it if you want to change the filters (date window, difficulties). Needs `HF_TOKEN` + accepting the dataset terms on HuggingFace once.

In [ ]:
RUN_PREPARE = False   # flip to True to rebuild the problem set from HuggingFace

import os
if RUN_PREPARE:
    !python prepare_lcb.py --min-date 2024-08-01 --difficulties easy,medium,hard --limit 0
else:
    n = sum(1 for _ in open("lcb_problems.jsonl"))
    print(f"using committed lcb_problems.jsonl ({n} problems)")

### Stage 2 · `extract_lcb_tests.py` → `lcb_tests.jsonl`  *(required for scoring)*
LCB's graded tests are HF-gated and **not** committed — and the file is **large (~400 MB+)** because private test cases are bulky. Pick the option that fits:

- **A · `HF_TOKEN`** — regenerate from HuggingFace, no file transfer. Add the secret, re-run this cell.
- **B · Google Drive** *(recommended if you already have the file)* — put `lcb_tests.jsonl` in your Drive once, mount, copy in. Fast and reliable for a file this size.
- **C · Browser upload** — works, but slow for ~400 MB and can time out. Note: `files.upload()` only transfers the file(s) you pick in the dialog — **not** folders or subtrees.

In [ ]:
import os
if os.path.exists("lcb_tests.jsonl"):
    print(f"lcb_tests.jsonl already present ({os.path.getsize('lcb_tests.jsonl')//1_000_000} MB)")
elif os.environ.get("HF_TOKEN"):
    !python extract_lcb_tests.py                          # Option A
else:
    print("Need lcb_tests.jsonl (LCB tests, HF-gated, ~400MB+). Choose one, then re-run Stage 5:")
    print(" A) add an HF_TOKEN secret and re-run THIS cell (regenerates it)")
    print(" B) Google Drive (recommended for the big file) — uncomment:")
    # from google.colab import drive; drive.mount("/content/drive")
    # import shutil; shutil.copy("/content/drive/MyDrive/lcb_tests.jsonl", "lcb_tests.jsonl")
    print(" C) browser upload (slow for ~400MB; picks only the file you select, not folders):")
    # from google.colab import files
    # up = files.upload()                                 # select ONLY lcb_tests.jsonl
    # name = next(iter(up)); open("lcb_tests.jsonl", "wb").write(up[name])

print("tests ready:", os.path.exists("lcb_tests.jsonl"))

### Stage 3 · `generate_lcb_prompts.py` → `lcb_v3_prompts.json`
Wraps each problem in the 4 framings (terse / casual / detailed / multilingual). The **mock** backend is deterministic and needs no API key.

In [ ]:
!python generate_lcb_prompts.py --backend mock

import json
prompts = json.load(open("lcb_v3_prompts.json"))
print(f"{len(prompts)} prompts (167 problems x 4 framings)\n")
print("example entry:")
e = prompts[0]
print(" task_id:", e["task_id"], "| condition:", e["condition"])
print(" prompt (first 400 chars):\n", e["prompt"][:400], "...")

### The heart of it · the extractor code
`score_lcb.py` turns a free-form model reply into a runnable program. These are the functions the paper is about — read them here before running them.

In [ ]:
import inspect, score_lcb
for fn in (score_lcb.extract_solution, score_lcb._trim_to_compilable,
           score_lcb.resolve_callable, score_lcb.passes):
    print("="*78)
    print(inspect.getsource(fn))

#### View any script file
Clicking a `.py` in Colab's **Files** pane often shows nothing — that viewer is unreliable. Read code from a cell instead. Change the path to any script; syntax-highlighted and read-only.

In [ ]:
from IPython.display import Code
Code(filename="score_lcb.py", language="python")   # e.g. "../vibe_tax_v2/run_vibe_tax.py", "generate_lcb_prompts.py"
# alternatives:  !cat score_lcb.py       (plain text)
#                %load score_lcb.py       (loads the file INTO this cell to read/run)

## Line-by-line walkthrough of the extractor
The four functions the extraction paper is about, annotated per line. Read top to
bottom; each runs during Stage 5 for every model completion. Chain per completion:
`extract_solution` (get code) -> exec + `resolve_callable` (find the function) ->
per test compare -> `passes` (per-test timeout + all-must-pass verdict).

### (1) `_trim_to_compilable` -- strip trailing prose so correct code parses
```python
def _trim_to_compilable(code):
    lines = code.split("\n")     # split the reply text into individual lines
    while lines:                  # loop while any lines remain
        src = "\n".join(lines)   # rejoin the current lines back into one string
        try:
            ast.parse(src)        # can Python PARSE this (syntax check, does NOT run it)?
            return src            # yes -> this is the largest valid-Python prefix; return it
        except SyntaxError:
            lines.pop()           # no -> drop the LAST line (trailing prose) and retry
    return None                   # nothing ever parsed -> no usable code
```
**Why:** replies are often `...correct code... + an English paragraph`. The paragraph
makes `ast.parse` throw, so we peel lines off the bottom until the rest parses —
keeping the code, dropping the prose. `ast.parse` never executes, so it is side-effect free.

### (2) `extract_solution` -- pull runnable code out of a messy reply
```python
def extract_solution(completion, entry):
    if not completion:                       # empty/None reply
        return None                          #   -> nothing to extract
    text = "\n".join(l for l in completion.split("\n")
                      if not l.strip().startswith("```"))   # copy w/ fence MARKERS removed
    blocks = re.findall(r"```(?:python|py)?\s*\n(.*?)```", completion, re.DOTALL)  # fenced blocks
    code_blocks = [b for b in blocks
                   if re.search(r"(?m)^\s*(?:class|def|import|from|@)\s", b)]  # keep code-like blocks

    cands = []                               # ordered list of candidate programs to try
    m = re.search(r"(?m)^(?:class|def|import|from|@)\s", text)  # first top-level construct
    if m:
        cands.append(text[m.start():])       # (B) slice from there -> KEEPS helpers above Solution
    if code_blocks:
        cands.append("\n\n".join(code_blocks))  # (A) all code blocks joined (helper+Solution split)
    for b in blocks:                         # (C) any single block naming the target
        if "class Solution" in b or f"def {entry}" in b:
            cands.append(b)
    for anchor in ("class Solution", f"def {entry}"):   # (D) last resort: slice from the anchor
        i = text.find(anchor)
        if i != -1:
            cands.append(text[i:])

    for c in cands:                          # try candidates IN ORDER
        t = _trim_to_compilable(c)           #   trim each to its compilable prefix
        if t and ("class Solution" in t or f"def {entry}" in t):  # compiles AND still has target
            return t                         #   -> accept the first that qualifies
    return cands[0] if cands else None       # graceful fallback (never a spurious None)
```
**Key property:** it never invents correctness — whatever it returns still has to pass
the real tests. Candidate (B) is tried first because it preserves helper classes.

### (3) `resolve_callable` -- find the function whatever container it's in
```python
def resolve_callable(ns, entry):             # ns = namespace after exec'ing the code
    import inspect
    f = ns.get(entry)                         # (1) a top-level function literally named entry?
    if callable(f) and not inspect.isclass(f):
        return f                              #     yes -> bare-function shape, return it
    classes = [v for v in ns.values() if inspect.isclass(v)]        # all top-level classes
    nested  = [v for c in classes for v in vars(c).values()
               if inspect.isclass(v)]                                # classes nested 1 level in
    cands = [c for c in (classes + nested)
             if callable(getattr(c, entry, None))]                   # classes exposing entry()
    if not cands:
        return None                           # entry not found in any shape
    cands.sort(key=lambda c: c.__name__ != "Solution")  # prefer a class named 'Solution'
    try:
        inst = cands[0]()                     # instantiate (LCB Solution takes no args)
    except Exception:
        return None
    return getattr(inst, entry, None)         # return the BOUND method
```
**Why:** a correct answer is sometimes a bare function, a renamed class, or a nested
class (a code smell, not a wrong answer). This binds to the real implementation anyway.

### (4) `_worker` + `passes` -- run the tests safely, per-test timeout
```python
def _worker(code, entry, tests, q):          # runs in a SEPARATE process
    ns = {}
    try:
        exec(_IMPORTS + code, ns)             # run the code (+common imports) into ns
    except Exception:
        q.append(False); return               # won't import -> one failure, stop
    fn = resolve_callable(ns, entry)          # get the entry function, any shape
    for t in tests:
        try:
            args = parse_input(t["input"])    # "1\n[2,3]" -> [1, [2,3]] (one literal per line)
            expected = parse_lit(t["output"]) # parse expected return as a Python literal
            ok = (fn is not None) and eq(fn(*args), expected)  # call + compare (float tolerant)
        except Exception:
            ok = False                        # any runtime error on this test = fail
        q.append(ok)                          # STREAM this test's result out immediately
        if not ok:
            return                            # early exit: one bad test fails the problem

def passes(code, entry, tests):
    if not code or not tests:
        return False
    mgr = multiprocessing.Manager(); q = mgr.list()   # list shared between processes
    p = multiprocessing.Process(target=_worker, args=(code, entry, tests, q))
    p.start()                                 # launch worker (a crash/loop can't kill us)
    seen = 0; last_progress = time.time()
    while p.is_alive():
        n = len(q)
        if n > seen:                          # worker reported new result(s)
            if q[n - 1] is False:             #   newest is a failure
                p.kill(); p.join(2); return False
            seen = n; last_progress = time.time()   # progress -> reset the per-test clock
        elif time.time() - last_progress > PER_TEST_TIMEOUT:  # no result for too long
            p.kill(); p.join(2); return False #   the current test hung (TLE) -> fail
        time.sleep(0.02)                      # poll ~50x/sec, don't busy-spin
    res = list(q)
    return len(res) == len(tests) and all(res)  # every test present AND all passed
```
**Why a subprocess + per-test clock:** each test independently gets `PER_TEST_TIMEOUT`
seconds, so a correct-but-slow solution on a many-test problem isn't failed by one
global budget; a hanging test is killed without taking down the notebook.

## Inspect ONE problem interactively
Run the cell below once to define `inspect_problem(...)`, then use the **call cell**
after it: change `task_id` (and optionally `condition` / `model` / `custom_input`) and
re-run to step through problems one at a time. Shows the extracted code, the resolved
function, an optional custom call, and the real graded tests (if `lcb_tests.jsonl` is present).

In [ ]:
import json, os, multiprocessing
import score_lcb
from score_lcb import (extract_solution, resolve_callable, parse_input,
                       parse_lit, eq, _IMPORTS, sample_tests)

# load the data once
PROBS = {json.loads(l)["task_id"]: json.loads(l) for l in open("lcb_problems.jsonl")}
RESP  = {(r["task_id"], r["level"], r["model"]): r
         for r in json.load(open("lcb_v3_responses.json"))}
TESTS = ({json.loads(l)["task_id"]: json.loads(l) for l in open("lcb_tests.jsonl")}
         if os.path.exists("lcb_tests.jsonl") else {})

def _run_one_test(code, entry, case, timeout=6):
    """Run ONE test in a subprocess; return (status, got_repr). Safe against hangs."""
    def worker(q):
        ns = {}
        try:
            exec(_IMPORTS + code, ns)
        except Exception as e:
            q.append(("error", f"exec failed: {type(e).__name__}: {e}")); return
        fn = resolve_callable(ns, entry)
        if fn is None:
            q.append(("error", "entry not found")); return
        try:
            got = fn(*parse_input(case["input"]))
            q.append(("pass" if eq(got, parse_lit(case["output"])) else "wrong", repr(got)))
        except Exception as e:
            q.append(("error", f"{type(e).__name__}: {e}"))
    mgr = multiprocessing.Manager(); q = mgr.list()
    p = multiprocessing.Process(target=worker, args=(q,)); p.start(); p.join(timeout)
    if p.is_alive():
        p.kill(); p.join(1); return ("timeout", f">{timeout}s (loop/TLE)")
    return q[0] if q else ("error", "worker crashed")

def inspect_problem(task_id, condition="agentic_terse", model="chatgpt",
                    n_show=None, only_fails=False, width=40, max_tests=60,
                    custom_input=None, show_raw=False):
    p = PROBS.get(task_id)
    if not p:
        print("unknown task_id:", task_id, "| e.g.", list(PROBS)[:3]); return
    entry = p["entry_point"]
    print(f"TASK {task_id} | difficulty={p.get('difficulty')} | entry={entry}")
    print(f"condition={condition} | model={model}")
    print("-"*72)
    print("PROBLEM (first 500 chars):\n", p["question_content"][:500], "...\n")
    r = RESP.get((task_id, condition, model))
    if not r:
        print("no recorded completion for that (task, condition, model).")
        print("  conditions here:", sorted({k[1] for k in RESP if k[0]==task_id}))
        print("  models here    :", sorted({k[2] for k in RESP if k[0]==task_id}))
        return
    if show_raw:
        print("RAW MODEL REPLY:\n", r["completion"], "\n" + "-"*72)
    code = extract_solution(r["completion"], entry)
    print("EXTRACTED CODE:\n", code, "\n" + "-"*72)
    ns = {}
    try:
        exec(_IMPORTS + code, ns)
        fn = resolve_callable(ns, entry)
        print("RESOLVED CALLABLE:", fn)
    except Exception as e:
        print("EXTRACTED CODE FAILS TO EXEC:", type(e).__name__, e); return
    if custom_input is not None:              # call the solution with YOUR OWN input
        try:
            print(f"\nCUSTOM INPUT {custom_input!r}  ->  {fn(*parse_input(custom_input))!r}")
        except Exception as e:
            print("custom call error:", type(e).__name__, e)
    rec = TESTS.get(task_id)
    if not rec:
        print("\n(no lcb_tests.jsonl -> graded tests skipped; run Stage 2 to enable)")
        return
    from collections import Counter
    cases = sample_tests(rec, max_tests)
    tally = Counter()
    print(f"\nGRADED TESTS ({len(cases)} run"
          + (", showing only failures" if only_fails else "") + "):")
    for i, c in enumerate(cases):
        status, got = _run_one_test(code, entry, c)
        tally[status] += 1
        if (n_show is None or i < n_show) and (not only_fails or status != "pass"):
            inp = c["input"] if isinstance(c["input"], str) else str(c["input"])
            inp = inp.replace("\n", " | ")                     # flatten multi-line input to one row
            print(f"  #{i:3d} {status:7s} in={inp[:width]:<{width}} "
                  f"exp={str(c['output'])[:width]:<{width}} got={got[:width]}")
    npass = tally.get("pass", 0)
    print("\n  tally: " + "  ".join(f"{k}={v}" for k, v in sorted(tally.items())))
    print(f"  RESULT: {npass}/{len(cases)} passed  ->  {'PASS' if npass==len(cases) else 'FAIL'}")

print("inspect_problem() ready. Edit the call cell below and re-run.")

### ▶ Call cell — edit and re-run per problem

In [ ]:
inspect_problem(
    task_id   = "lcb/3517",          # <-- change to any problem id (browse the list below)
    condition = "agentic_terse",     # agentic_terse | agentic_casual | webchat_detailed | webchat_multilingual
    model     = "chatgpt",           # chatgpt | claude | codestral
    n_show    = None,                # None = EVERY test row; or an int for just the first N
    only_fails = False,              # True = print only the tests that did NOT pass
    width     = 40,                  # column width for input / expected / got
    max_tests = 60,                  # how many tests to actually run (raise to run more)
    custom_input = None,             # e.g. "5\n[1,2,3]" to call the solution with your own args
    show_raw  = False,               # True to also print the raw model reply
)

### Browse available problem ids

In [ ]:
import json
print(f"{'task_id':10s} {'difficulty':10s} entry_point")
for l in list(open("lcb_problems.jsonl")):
    d = json.loads(l)
    print(f"{d['task_id']:10s} {d['difficulty']:10s} {d['entry_point']}")

### View the unit tests for one problem
See the actual LCB test cases (full input + expected output) for a `task_id`, filtered to
**all**, only the tests a solution **passed**, or only the ones it **failed** (pass/fail run
the chosen model's solution to classify). Needs `lcb_tests.jsonl`.

In [ ]:
import json

def show_tests(task_id, which="all", condition="agentic_terse", model="chatgpt",
               max_tests=60, full=True, width=80):
    """which = 'all' | 'pass' | 'fail'  (pass/fail run the model's solution to classify each test)."""
    rec = TESTS.get(task_id)
    if not rec:
        print("no tests for", task_id, "-- run Stage 2 (lcb_tests.jsonl) to enable"); return
    cases = sample_tests(rec, max_tests)
    npub = 0 if PRIVATE_ONLY else len(rec.get("public_tests", []))   # public tests come first
    need_run = which in ("pass", "fail")
    code = entry = None
    if need_run:
        r = RESP.get((task_id, condition, model))
        if not r:
            print(f"no completion for {task_id}/{condition}/{model} -> showing ALL instead")
            which, need_run = "all", False
        else:
            entry = r["entry_point"]; code = extract_solution(r["completion"], entry)
    print(f"TASK {task_id}: {len(cases)} LCB tests (public first, then private)"
          + (f"  |  filter={which} via {model}/{condition}" if need_run else f"  |  filter={which}"))
    shown = 0
    for i, c in enumerate(cases):
        status = got = None
        if need_run:
            status, got = _run_one_test(code, entry, c)
            if which == "pass" and status != "pass": continue
            if which == "fail" and status == "pass": continue
        tag = "public" if i < npub else "private"
        head = f"#{i} [{tag}]" + (f"  -> {status.upper()}" if status else "")
        print("\n" + head)
        inp = c["input"] if isinstance(c["input"], str) else json.dumps(c["input"])
        if full:
            print("  input:")
            for ln in str(inp).split("\n"):
                print("     ", ln)
            print("  expected:", c["output"])
            if need_run:
                print("  got     :", got)
        else:
            flat = str(inp).replace("\n", " | ")               # one-line input
            print(f"  in={flat[:width]}  exp={str(c['output'])[:width]}"
                  + (f"  got={got[:width]}" if need_run else ""))
        shown += 1
    print(f"\nshown {shown} of {len(cases)} test(s)")

print("show_tests() ready -- edit the call cell below.")

#### ▶ Call cell — view unit tests (edit and re-run)

In [ ]:
show_tests(
    task_id   = "lcb/3517",       # which problem's tests
    which     = "all",            # "all" | "pass" | "fail"
    condition = "agentic_terse",  # used only when which is pass/fail (to run the solution)
    model     = "chatgpt",        # used only when which is pass/fail
    max_tests = 60,               # how many tests to consider
    full      = True,             # True = full multi-line input/expected; False = one-line truncated
    width     = 80,
)

#### Same tests, as runnable `assert` code
LCB tests are stored as input/output **data**, not code. This renders each one as the
equivalent Python assertion (`Solution().entry(args) == expected`) — same all/pass/fail filter.

In [ ]:
def show_tests_as_code(task_id, which="all", condition="agentic_terse", model="chatgpt", max_tests=60):
    """Render each LCB input/output test as a runnable assert line."""
    rec = TESTS.get(task_id)
    if not rec:
        print("no tests for", task_id, "-- run Stage 2 (lcb_tests.jsonl)"); return
    entry = (PROBS.get(task_id, {}).get("entry_point")
             or next((r["entry_point"] for k, r in RESP.items() if k[0] == task_id), None))
    cls = PROBS.get(task_id, {}).get("class_name") or "Solution"
    cases = sample_tests(rec, max_tests)
    npub = 0 if PRIVATE_ONLY else len(rec.get("public_tests", []))
    need_run = which in ("pass", "fail")
    code = None
    if need_run:
        r = RESP.get((task_id, condition, model))
        if r:
            code = extract_solution(r["completion"], entry)
        else:
            print(f"no completion for {task_id}/{condition}/{model} -> showing ALL"); which, need_run = "all", False
    print(f"# LCB tests for {task_id} as assertions  ->  {cls}().{entry}(...)")
    shown = 0
    for i, c in enumerate(cases):
        status = None
        if need_run:
            status, _ = _run_one_test(code, entry, c)
            if which == "pass" and status != "pass": continue
            if which == "fail" and status == "pass": continue
        args = parse_input(c["input"]); expected = parse_lit(c["output"])
        arglist = ", ".join(repr(a) for a in args)
        note = "  # " + ("public" if i < npub else "private") + (f", {status}" if status else "")
        print(f"assert {cls}().{entry}({arglist}) == {expected!r}{note}")
        shown += 1
    print(f"\n# shown {shown} of {len(cases)} test(s)")

show_tests_as_code("lcb/3517", which="all")

### Read the full problem statement
Insert a task number to see the complete LCB problem — description, worked **examples**
(input/output/explanation), and constraints — plus the function signature.

In [ ]:
def show_problem(task_id):
    """Print the FULL LCB problem statement (description + examples + constraints) and signature."""
    p = PROBS.get(task_id)
    if not p:
        r = next((v for k, v in RESP.items() if k[0] == task_id), None)
        if not r:
            print("unknown task_id:", task_id, "| e.g.", sorted({k[0] for k in RESP})[:5]); return
        print(f"TASK {task_id} | entry={r['entry_point']}")
        print("(no lcb_problems.jsonl loaded -> showing the wrapped prompt that was sent)\n" + "="*72)
        print(r.get("prompt_text", "")); return
    print(f"TASK {task_id}")
    print(f"  difficulty : {p.get('difficulty')}")
    print(f"  entry_point: {p.get('entry_point')}({p.get('params','')})")
    print(f"  class      : {p.get('class_name','Solution')}")
    print(f"  contest    : {p.get('contest_date')}")
    print("="*72)
    print("PROBLEM STATEMENT (full -- description, examples, constraints):\n")
    print(p.get("question_content", ""))
    if p.get("starter_code"):
        print("\n" + "="*72 + "\nSTARTER CODE (signature the solution must match):\n")
        print(p["starter_code"])

show_problem("lcb/3517")     # <-- change the task number and re-run

### Stage 4 · `run_vibe_tax.py` → `lcb_v3_responses.json`  *(OPTIONAL — costs money)*
The model responses are **already committed**, so leave this off to score them as-is. Flip `RUN_API = True` only to re-query (needs API keys and spends tokens). It resumes from a progress file, so a partial run is safe.

In [ ]:
RUN_API = False   # True re-queries the models ($$$). Default: use committed responses.

import os
if RUN_API:
    os.environ["PROMPTS_FILE"]  = os.path.abspath("lcb_v3_prompts.json")
    os.environ["OUTPUT_FILE"]   = os.path.abspath("lcb_v3_responses.json")
    os.environ["PROGRESS_FILE"] = os.path.abspath("lcb_v3_progress.json")
    # os.environ["ONLY_MODELS"] = "chatgpt"   # optional: one provider
    !python ../vibe_tax_v2/run_vibe_tax.py
else:
    import json
    r = json.load(open("lcb_v3_responses.json"))
    print(f"using committed lcb_v3_responses.json ({len(r)} completions)")

### Stage 5 · `score_lcb.py` → `lcb_scored.json`  ← **the extractor runs here**
For every completion: `extract_solution()` pulls runnable code, `resolve_callable()` binds the entry point in any shape, and `passes()` runs each test under a per-test timeout. Writes a `passed` bool per record plus aggregate stats.

In [ ]:
import os, json
if not os.path.exists("lcb_tests.jsonl"):
    print("Cannot score without lcb_tests.jsonl — run Stage 2 first.")
else:
    !python score_lcb.py --max-tests 60
    stats = json.load(open("lcb_scored_stats.json"))
    print("\nOVERALL: {pass_rate}%  (n={total})".format(**stats["overall"]))
    print("by condition:")
    for k,v in stats["by_condition"].items():
        print(f"  {k:22s} {v['pass_rate']}%")
    print("by difficulty:")
    for k,v in stats["by_difficulty"].items():
        print(f"  {k:8s} {v['pass_rate']}%")

### Stage 6a · `mcnemar_lcb.py` → `lcb_mcnemar_stats.json`
Paired within-problem test of terse vs each other framing (the framing-null result).

In [ ]:
import json
!python mcnemar_lcb.py --base agentic_terse
m = json.load(open("lcb_mcnemar_stats.json"))
print("\nterse vs each framing (capable-models slice):")
for pair, slices in m.items():
    cm = slices.get("capable_models")
    if cm:
        print(f"  {pair:44s} delta={cm['delta_pts']:+.1f} pts  p={cm['p']}  "
              f"{'sig' if cm['significant'] else 'n.s.'}")

### Stage 6b–d · deeper analyses  *(optional; some are heavy)*
Partial credit re-runs failed attempts (slow), capability slices by topic, and the Excel workbook.

In [ ]:
RUN_PARTIAL   = False   # slow: re-runs every failed attempt for per-test %
RUN_CAPABILITY = True
RUN_WORKBOOK   = True

if RUN_PARTIAL:
    !python full_partial_credit.py --timeout 6
if RUN_CAPABILITY:
    !python analyze_lcb_capability.py 2>/dev/null || echo "(analyze_lcb_capability.py needs lcb_scored.json)"
if RUN_WORKBOOK:
    !python ../../build_breakdown.py
    print("\nWorkbook written to repo root: The_Vibe_Tax_Problem_Breakdown.xlsx")
    # from google.colab import files; files.download("../../The_Vibe_Tax_Problem_Breakdown.xlsx")

### Inspect / debug side-paths (no full run)
- `print_prompt.py <task_id>` — the exact prompt that would be sent.
- `manual_test.py` — paste a reply, run `extract_solution` + tests on that one completion (the tool that surfaced the extractor bugs).
- `test_class_shape.py` — unit-tests the shape resolver and scans the corpus for shape-mismatch prevalence.

In [ ]:
# see the extractor on a single completion, in isolation:
import json, score_lcb
r = json.load(open("lcb_v3_responses.json"))[0]
code_out = score_lcb.extract_solution(r["completion"], r["entry_point"])
print("task:", r["task_id"], "| entry:", r["entry_point"])
print("extracted code (first 300 chars):\n", (code_out or "")[:300])

print("\n--- shape-resolver unit test + corpus prevalence scan ---")
!python test_class_shape.py

---
**Recap.** Stages 1–4 build the inputs (problems, tests, prompts, responses); Stage 5 is where the extractor turns replies into pass/fail; Stage 6 analyses the scores. Re-scoring is decoupled from the API — once `lcb_v3_responses.json` exists, changing the extractor and re-running **Stage 5 only** (no model calls) is how every fix in the paper was validated.